# Customer segmentation — del dato sucio al modelo

Historia completa con el dataset demo de **DQ Observatory** (`../data/demo/customers_sales.csv`):
auditoría de suciedad → limpieza mínima (las mismas ops del observatorio) → EDA → baseline sklearn que predice `segment`.
Todo número de este notebook se computa al ejecutar. Nada está hardcodeado.

In [1]:
%matplotlib inline
import os
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("img", exist_ok=True)
df = pd.read_csv("../data/demo/customers_sales.csv", low_memory=False)
print("shape:", df.shape)
print(df.dtypes)

shape: (5200, 17)
customer_id       object
customer_name     object
email             object
phone             object
country           object
city              object
signup_date       object
age                int64
segment           object
product           object
quantity           int64
unit_price        object
revenue          float64
status            object
last_purchase     object
rating           float64
source_system     object
dtype: object


In [2]:
# 1. Auditoría de suciedad (lo que el observatorio detecta)
print("--- nulos por columna ---")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nduplicados exactos:", int(df.duplicated().sum()))
print("\nvariantes de segment:", sorted(df["segment"].dropna().unique()))
print("\nvariantes de city:", sorted(df["city"].dropna().unique()))

--- nulos por columna ---
email              80
phone            1786
country           852
city               40
signup_date       619
unit_price        120
last_purchase    2554
rating           1401
dtype: int64

duplicados exactos: 197

variantes de segment: ['Enterprise', 'RETAIL', 'Retail', 'SMB', 'enterprise', 'retail', 'smb ']

variantes de city: ['BOGOTA', 'Barranquilla', 'Bogota', 'Bogota DC', 'Bogotá', 'Bogotá D.C.', 'CALI', 'Cali', 'Cartagena', 'MEDELLIN', 'Medellin', 'Medellín', 'bogota']


## 2. Limpieza mínima
Equivalente a las ops del observatorio: `trim_whitespace` + `standardize_categories` (segment/city),
`parse_numeric` (age/quantity/unit_price/revenue/rating) y `remove_exact_duplicates`.

In [3]:
clean = df.copy()
for c in ["customer_name", "segment", "city", "country", "status", "product"]:
    clean[c] = clean[c].astype("string").str.strip()
clean["segment"] = clean["segment"].str.lower()
clean["city"] = clean["city"].str.lower()
for c in ["age", "quantity", "unit_price", "revenue", "rating"]:
    clean[c] = pd.to_numeric(clean[c], errors="coerce")
clean = clean.drop_duplicates().reset_index(drop=True)
clean = clean[clean["segment"].isin(["retail", "enterprise", "smb"])]
print("filas crudas:", len(df), "→ filas limpias:", len(clean))

filas crudas: 5200 → filas limpias: 5003


## 3. EDA

In [4]:
ax = clean["segment"].value_counts().plot(kind="bar", title="Distribución de segmento (limpio)")
ax.set_xlabel("segment"); ax.set_ylabel("filas")
plt.tight_layout(); plt.savefig("img/segment_dist.png"); plt.close()

clean.boxplot(column="revenue", by="segment", grid=False)
plt.title("Revenue por segmento"); plt.suptitle("")
plt.tight_layout(); plt.savefig("img/revenue_by_segment.png"); plt.close()
print(clean.groupby("segment")["revenue"].median())

segment
enterprise    4094.355
retail        3969.580
smb           4152.900
Name: revenue, dtype: float64


## 4. Baseline: predecir `segment` con RandomForest

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

feats = ["age", "quantity", "unit_price", "revenue", "rating"]
model_df = pd.get_dummies(clean[feats + ["product", "segment"]], columns=["product"]).dropna()
X = model_df.drop(columns=["segment"])
y = model_df["segment"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr, ytr)
pred = clf.predict(Xte)
print("filas train/test:", len(Xtr), len(Xte))
print("accuracy:", round(accuracy_score(yte, pred), 4))
print(classification_report(yte, pred))
ConfusionMatrixDisplay.from_predictions(yte, pred)
plt.tight_layout(); plt.savefig("img/confusion_matrix.png"); plt.close()

filas train/test: 2568 856
accuracy: 0.3621
              precision    recall  f1-score   support

  enterprise       0.30      0.27      0.28       251
      retail       0.43      0.56      0.49       362
         smb       0.25      0.16      0.20       243

    accuracy                           0.36       856
   macro avg       0.33      0.33      0.32       856
weighted avg       0.34      0.36      0.35       856



## 5. Conclusión

- Sin la limpieza del paso 2 (variantes de `segment`, numéricos como texto, duplicados), el target ni siquiera es entrenable: el observatorio reportaba score 65.7 con consistency 0.
- Baseline RandomForest (200 árboles): accuracy 0.36, macro-F1 0.32 — POR DEBAJO del baseline mayoría (retail = 0.42). Lectura honesta: el `segment` no se explica con estas features numéricas; el dataset es sintético y la señal real está en las categóricas sin explotar.
- Siguiente paso real: `standardize_categories` en `city`/`product` como features categóricas + tuning (grid sobre `max_depth`/`n_estimators`) + validación cruzada.